# Day 1: Data Profiling and Chunking
This notebook demonstrates downloading the 25GB Kaggle dataset using the Kaggle API
directly into Databricks Volumes and splitting it into chunks for processing.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *

In [0]:
CATALOG = "vstone"
SCHEMA = "bronze"
RAW_VOLUME = "/Volumes/vstone/bronze/raw_volume"

In [0]:
flight_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(f"{RAW_VOLUME}/Combined_Flights*.csv")
)

In [0]:
display(flight_df)
print("Rows: ", flight_df.count())
print("Columns: ", len(flight_df.columns))

In [0]:
display( flight_df.dtypes)

In [0]:
null_df = flight_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in flight_df.columns
])

display(null_df)

In [0]:
duplicates = flight_df.count() - flight_df.dropDuplicates().count()

print("Duplicate Records :", duplicates)

In [0]:
print(
    flight_df.select("Airline").distinct().count()
)

In [0]:
print (
    "Distinct Origin Airports:",
    flight_df.select("Origin").distinct().count()
)

In [0]:
print (
    "Distinct Destination Airports:",
    flight_df.select("Dest").distinct().count()
)

In [0]:
flight_df.select(F.min("FlightDate"), F.max("FlightDate")).show()

In [0]:
display(
    flight_df.select(
        "DepDelay",
        "ArrDelay", 
        "Distance",
        "AirTime"
    ).describe()
)

In [0]:
from pyspark.sql import functions as F

flight_df.groupBy("Year") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("Year") \
    .show()

In [0]:
chunk1_csv = flight_df.filter(F.col("Year") == 2018)

chunk2_csv = flight_df.filter(F.col("Year") == 2019)

chunk3_json = flight_df.filter(F.col("Year") == 2020)

chunk4_xml = flight_df.filter(F.col("Year").isin(2021, 2022))

In [0]:
print("Chunk 1 (2018 CSV):", chunk1_csv.count())

print("Chunk 2 (2019 CSV):", chunk2_csv.count())

print("Chunk 3 (2020 JSON):", chunk3_json.count())

print("Chunk 4 (2021-2022 XML):", chunk4_xml.count())

In [0]:
display(chunk1_csv.limit(5))
display(chunk2_csv.limit(5))
display(chunk3_json.limit(5))
display(chunk4_xml.limit(5))

In [0]:
chunk1_csv.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/vstone/bronze/raw_volume/csv_initial")

In [0]:
chunk2_csv.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/vstone/bronze/raw_volume/csv_incremental")

In [0]:
chunk3_json.write \
    .mode("overwrite") \
    .json("/Volumes/vstone/bronze/raw_volume/json")

In [0]:
chunk4_xml.write \
    .mode("overwrite") \
    .format("xml") \
    .option("rootTag", "Flights") \
    .option("rowTag", "Flight") \
    .save("/Volumes/vstone/bronze/raw_volume/xml")

In [0]:
print("CSV Initial Records:",
      spark.read.option("header", True)
      .csv("/Volumes/vstone/bronze/raw_volume/csv_initial")
      .count())

print("CSV Incremental Records:",
      spark.read.option("header", True)
      .csv("/Volumes/vstone/bronze/raw_volume/csv_incremental")
      .count())

print("JSON Records:",
      spark.read.json("/Volumes/vstone/bronze/raw_volume/json")
      .count())

In [0]:
display(dbutils.fs.ls("/Volumes/vstone/bronze/raw_volume/xml"))